In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import sys
import os
import numpy as np
# Add the parent directory so we can import from the 'src' folder
sys.path.append(os.path.abspath(os.path.join('..')))

from src.models.eegnet import EEGNet
from src.data_prep import get_cleaned_epochs
from src.data_prep import get_multi_subject_data
epochs_multi, event_id = get_multi_subject_data(subject_list=[1, 2, 3], training=True)


epochs_multi, event_id = get_multi_subject_data(subject_list=[1, 2, 3], training=True)

# 2. Get the raw data [Trials, Channels, Samples]
X_multi = epochs_multi.get_data()
raw_y_multi = epochs_multi.events[:, -1]

# 3. Apply the Label Map (0, 1, 2, 3)
unique_classes = np.unique(raw_y_multi)
label_map = {raw_code: i for i, raw_code in enumerate(unique_classes)}
y_multi = np.array([label_map[code] for code in raw_y_multi])

# 4. Reshape for the AI [Batch, 1, Channels, Samples]
if X_multi.ndim == 3:
    X_multi = X_multi[:, None, :, :]

# 5. Convert to PyTorch Tensors
X_train_multi = torch.tensor(X_multi, dtype=torch.float32)
y_train_multi = torch.tensor(y_multi, dtype=torch.long)

# 6. DEFINING THE MISSING VARIABLE: train_loader_multi
# This is what feeds the 'batch_X' and 'batch_y' into your loop
train_loader_multi = DataLoader(
    TensorDataset(X_train_multi, y_train_multi), 
    batch_size=32, 
    shuffle=True
)

print(f"Success! train_loader_multi is defined with {len(train_loader_multi)} batches.")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading and Scaling Subject 1...
Loading and Scaling Subject 2...
Loading and Scaling Subject 3...
Not setting metadata
144 matching events found
Applying baseline correction (mode: mean)


/Users/elijahakpan/developer/NeuralStream/src/data_prep.py:52: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs)


Successfully combined 3 subjects.
Loading and Scaling Subject 1...
Loading and Scaling Subject 2...
Loading and Scaling Subject 3...
Not setting metadata
144 matching events found
Applying baseline correction (mode: mean)
Successfully combined 3 subjects.
Success! train_loader_multi is defined with 5 batches.


/Users/elijahakpan/developer/NeuralStream/src/data_prep.py:52: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs)


In [2]:
# Use the modular function we created to get the clean data
epochs, event_id = get_cleaned_epochs(subject_id=1)

# Convert MNE epochs to a NumPy array [Trials, Channels, TimePoints]
X = epochs.get_data() 
y = epochs.events[:, -1] - 769  # Normalize labels to start at 0


In [3]:
import numpy as np

# 1. Get the raw event codes
raw_y = epochs.events[:, -1]

# 2. Identify the unique classes (e.g., 769, 770, 771, 772)
unique_classes = np.unique(raw_y)
print(f"Raw classes found in data: {unique_classes}")

# 3. Create a mapping to 0, 1, 2, 3
# This maps the smallest ID to 0, next to 1, etc.
label_map = {raw_code: i for i, raw_code in enumerate(unique_classes)}
print(f"Mapping labels to: {label_map}")

# 4. Apply the mapping
y = np.array([label_map[code] for code in raw_y])

# 5. Convert to Tensor
y_tensor = torch.tensor(y, dtype=torch.long)

# Now proceed to create your DataLoader...


Raw classes found in data: [1 2 3 4]
Mapping labels to: {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2, np.int64(4): 3}


In [4]:
# Add the '1' dimension for the Convolutional layers
if X.ndim == 3:
    X = X[:, None, :, :]

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Create the DataLoader for batching
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)



In [5]:
# 1. Initialize the model with dynamic dimensions
model = EEGNet(nb_classes=len(event_id), Chans=X.shape[2], Samples=X.shape[-1])

# 1. New Math Components
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Learning Rate Scheduler: Reduces LR when accuracy stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 1. Stronger Noise (Increase from 0.01 to 0.1)
noise_level = 0.1 

# 2. Add a 'Best Loss' tracker for Early Stopping
best_loss = float('inf')
patience_counter = 0
early_stop_patience = 10

print("Starting Defensive Training...")
for epoch in range(100):
    model.train()
    epoch_loss = 0
    
    for batch_X, batch_y in train_loader_multi:
        # Add stronger noise to break memorization
        noise = torch.randn_like(batch_X) * noise_level
        batch_X_noisy = batch_X + noise
        
        optimizer.zero_grad()
        outputs = model(batch_X_noisy)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader_multi)
    scheduler.step(avg_loss)
    
    # --- EARLY STOPPING LOGIC ---
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        # Save the "Best" version of the model
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        
    if patience_counter >= early_stop_patience:
        print(f"Early stopping at epoch {epoch}. The model started over-memorizing.")
        break
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {avg_loss:.4f}, LR: {optimizer.param_groups[0]['lr']}")

# Load the best version before testing
model.load_state_dict(torch.load('best_model.pth'))
print("Best model loaded for evaluation.")


%load_ext autoreload
%autoreload 2


Starting Defensive Training...
Epoch 0, Loss: 1.4276, LR: 0.001
Epoch 5, Loss: 1.3353, LR: 0.001
Epoch 10, Loss: 1.1841, LR: 0.001
Epoch 15, Loss: 1.1173, LR: 0.001
Epoch 20, Loss: 0.9088, LR: 0.001
Epoch 25, Loss: 0.9714, LR: 0.001
Epoch 30, Loss: 0.8922, LR: 0.0005
Epoch 35, Loss: 0.9042, LR: 0.00025
Early stopping at epoch 38. The model started over-memorizing.
Best model loaded for evaluation.


In [6]:
model.eval() # Set the model to evaluation mode
correct = 0
total = 0

with torch.no_grad(): # No need to calculate gradients for evaluation
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1) # Get the highest probability class
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the training set: {accuracy:.2f}%')


Accuracy on the training set: 68.75%


In [7]:
# 1. Load the Evaluation data (Set training=False)
epochs_test, _ = get_cleaned_epochs(subject_id=1, training=False)

# 2. Convert to NumPy (This is where your previous error was)
X_test = epochs_test.get_data() 

# 3. Label Mapping (Use the SAME map from your training cell)
y_test = np.array([label_map[code] for code in epochs_test.events[:, -1]])

# 4. Prepare Tensors
if X_test.ndim == 3:
    X_test = X_test[:, None, :, :]

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print(f"Test data loaded: {X_test_tensor.shape[0]} trials.")


Test data loaded: 48 trials.


In [8]:
# 1. Initialize the model with dynamic dimensions
model_multi = EEGNet(nb_classes=len(event_id), Chans=X.shape[2], Samples=X.shape[-1])

# 1. New Math Components
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Learning Rate Scheduler: Reduces LR when accuracy stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 1. Stronger Noise (Increase from 0.01 to 0.1)
noise_level = 0.1 

# 2. Add a 'Best Loss' tracker for Early Stopping
best_loss = float('inf')
patience_counter = 0
early_stop_patience = 10

print("Starting Defensive Training...")
for epoch in range(100):
    model_multi.train()
    epoch_loss = 0
    
    for batch_X, batch_y in train_loader_multi:
        # Add stronger noise to break memorization
        noise = torch.randn_like(batch_X) * noise_level
        batch_X_noisy = batch_X + noise
        
        optimizer.zero_grad()
        outputs = model_multi(batch_X_noisy)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader_multi)
    scheduler.step(avg_loss)
    
    # --- EARLY STOPPING LOGIC ---
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        # Save the "Best" version of the model
        torch.save(model_multi.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        
    if patience_counter >= early_stop_patience:
        print(f"Early stopping at epoch {epoch}. The model started over-memorizing.")
        break
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {avg_loss:.4f}, LR: {optimizer.param_groups[0]['lr']}")

# Load the best version before testing
model_multi.load_state_dict(torch.load('best_model.pth'))
print("Best model loaded for evaluation.")


%load_ext autoreload
%autoreload 2


Starting Defensive Training...
Epoch 0, Loss: 1.4280, LR: 0.001
Epoch 5, Loss: 1.4379, LR: 0.001
Epoch 10, Loss: 1.4349, LR: 0.0005
Early stopping at epoch 13. The model started over-memorizing.
Best model loaded for evaluation.
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs.data, 1)
    test_total = y_test_tensor.size(0)
    test_correct = (predicted == y_test_tensor).sum().item()

test_accuracy = 100 * test_correct / test_total
print(f'REAL-WORLD ACCURACY (Unseen Session): {test_accuracy:.2f}%')


REAL-WORLD ACCURACY (Unseen Session): 41.67%


In [10]:
# 1. Load data from 3 different people to prevent overfitting
epochs_multi, event_id = get_multi_subject_data(subject_list=[1, 2, 3], training=True)

# 2. Convert to NumPy
X_multi = epochs_multi.get_data()
raw_y_multi = epochs_multi.events[:, -1]

# 3. Create the Label Map
unique_classes = np.unique(raw_y_multi)
label_map = {raw_code: i for i, raw_code in enumerate(unique_classes)}
y_multi = np.array([label_map[code] for code in raw_y_multi])

# 4. Prepare Tensors
if X_multi.ndim == 3:
    X_multi = X_multi[:, None, :, :]

X_train_multi = torch.tensor(X_multi, dtype=torch.float32)
y_train_multi = torch.tensor(y_multi, dtype=torch.long)

# 5. Create DataLoader
train_loader_multi = DataLoader(TensorDataset(X_train_multi, y_train_multi), batch_size=32, shuffle=True)

print(f"Total Training Samples: {X_train_multi.shape[0]}")



Loading and Scaling Subject 1...
Loading and Scaling Subject 2...
Loading and Scaling Subject 3...
Not setting metadata
144 matching events found
Applying baseline correction (mode: mean)
Successfully combined 3 subjects.
Total Training Samples: 144


/Users/elijahakpan/developer/NeuralStream/src/data_prep.py:52: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  combined_epochs = mne.concatenate_epochs(all_epochs)


In [11]:
# Initialize a FRESH model for the multi-subject data
model_multi = EEGNet(nb_classes=len(event_id), Chans=X_multi.shape[2], Samples=X_multi.shape[-1])
# 1. New Math Components
optimizer = torch.optim.Adam(model_multi.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Learning Rate Scheduler: Reduces LR when accuracy stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 1. Stronger Noise (Increase from 0.01 to 0.1)
noise_level = 0.1 

# 2. Add a 'Best Loss' tracker for Early Stopping
best_loss = float('inf')
patience_counter = 0
early_stop_patience = 10

print("Starting Defensive Training...")
for epoch in range(100):
    model_multi.train()
    epoch_loss = 0
    
    for batch_X, batch_y in train_loader_multi:
        # Add stronger noise to break memorization
        noise = torch.randn_like(batch_X) * noise_level
        batch_X_noisy = batch_X + noise
        
        optimizer.zero_grad()
        outputs = model_multi(batch_X_noisy)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader_multi)
    scheduler.step(avg_loss)
    
    # --- EARLY STOPPING LOGIC ---
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        # Save the "Best" version of the model
        torch.save(model_multi.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        
    if patience_counter >= early_stop_patience:
        print(f"Early stopping at epoch {epoch}. The model started over-memorizing.")
        break
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {avg_loss:.4f}, LR: {optimizer.param_groups[0]['lr']}")

# Load the best version before testing
model_multi.load_state_dict(torch.load('best_model.pth'))
print("Best model loaded for evaluation.")



Starting Defensive Training...
Epoch 0, Loss: 1.4126, LR: 0.001
Epoch 5, Loss: 1.3777, LR: 0.001
Epoch 10, Loss: 1.2661, LR: 0.001
Epoch 15, Loss: 1.1558, LR: 0.001
Epoch 20, Loss: 1.0076, LR: 0.001
Epoch 25, Loss: 0.9570, LR: 0.001
Epoch 30, Loss: 0.9667, LR: 0.001
Epoch 35, Loss: 0.8882, LR: 0.001
Epoch 40, Loss: 0.8796, LR: 0.0005
Epoch 45, Loss: 0.8924, LR: 0.0005
Epoch 50, Loss: 0.8188, LR: 0.0005
Epoch 55, Loss: 0.7866, LR: 0.00025
Epoch 60, Loss: 0.9162, LR: 0.00025
Epoch 65, Loss: 0.8259, LR: 0.000125
Early stopping at epoch 66. The model started over-memorizing.
Best model loaded for evaluation.


In [12]:
model_multi.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    # Use the X_test_tensor you created in the previous section
    outputs = model_multi(X_test_tensor)
    _, predicted = torch.max(outputs.data, 1)
    test_total = y_test_tensor.size(0)
    test_correct = (predicted == y_test_tensor).sum().item()

new_test_accuracy = 100 * test_correct / test_total
print(f'NEW REAL-WORLD ACCURACY: {new_test_accuracy:.2f}%')
print(f'Improvement: {new_test_accuracy - 35.42:.2f}%')


NEW REAL-WORLD ACCURACY: 37.50%
Improvement: 2.08%


In [13]:
# --- FINAL TEST: UNIVERSAL DECODING ---
# We already trained on Subjects 1, 2, and 3. 
# Let's see if the model can decode Subject 2's UNSEEN data.

def evaluate_new_subject(sub_id):
    # 1. Load the test session for the new subject
    epochs_test_sub, _ = get_cleaned_epochs(subject_id=sub_id, training=False)
    
    # 2. Prepare the data
    X_sub = epochs_test_sub.get_data()
    if X_sub.ndim == 3: 
        X_sub = X_sub[:, None, :, :]
    
    # 3. Use the same label map
    y_sub = np.array([label_map[code] for code in epochs_test_sub.events[:, -1]])
    
    X_sub_tensor = torch.tensor(X_sub, dtype=torch.float32)
    y_sub_tensor = torch.tensor(y_sub, dtype=torch.long)
    
    # 4. Evaluate
    model_multi.eval()
    with torch.no_grad():
        outputs = model_multi(X_sub_tensor)
        _, predicted = torch.max(outputs.data, 1)
        acc = 100 * (predicted == y_sub_tensor).sum().item() / y_sub_tensor.size(0)
    
    return acc

# Run the evaluation for the other subjects
acc_sub2 = evaluate_new_subject(sub_id=2)
acc_sub3 = evaluate_new_subject(sub_id=3)

print(f'Subject 2 Real-World Accuracy: {acc_sub2:.2f}%')
print(f'Subject 3 Real-World Accuracy: {acc_sub3:.2f}%')


Subject 2 Real-World Accuracy: 43.75%
Subject 3 Real-World Accuracy: 39.58%
